# ⚡ Twitch AutoClipper — Motor GPU Turbo (Google AI Ultra + 20 TB Drive)

Este notebook conecta diretamente na mesma pasta **`TwitchAutoClipper`** dos seus **20 TB do Google Drive** usada pelo GitHub Actions, destravando toda a potência da sua assinatura **Google AI Ultra**:
- **GPUs `L4` / `A100` + `NVENC`**: renderização de lotes de 8–15 cortes em poucos segundos.
- **`faster-whisper large-v3` (CUDA float16)**: transcrição local de altíssima precisão sem depender de limites externos.
- **`Demucs` na GPU**: remoção de músicas com direitos autorais (DMCA) em ~5 segundos por corte.
- **Execução em Segundo Plano (*Background Execution* até 24h)**: ideal para maratonas/subathons podendo fechar a aba do navegador.

In [ ]:
#@title 1. Montar Google Drive (20 TB) e Verificar GPU L4/A100
import os
import subprocess
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/TwitchAutoClipper'
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.environ['GDRIVE_MOUNT_PATH'] = DRIVE_ROOT

print(f'✅ Google Drive conectado em: {DRIVE_ROOT}')
print('\n🎮 Placa de vídeo alocada no Google AI Ultra:')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], check=False)

In [ ]:
#@title 2. Clonar/Atualizar Repositório e Instalar Motores GPU (`faster-whisper` + `demucs`)
REPO_URL = 'https://github.com/philipemarcel/autoclipper.git' #@param {type:"string"}
BRANCH = 'main' #@param {type:"string"}

import os
if not os.path.exists('/content/twitch-autocliper'):
    !git clone -b {BRANCH} {REPO_URL} /content/twitch-autocliper
else:
    %cd /content/twitch-autocliper
    !git pull origin {BRANCH}

%cd /content/twitch-autocliper
!apt-get update -qq && apt-get install -y -qq ffmpeg fonts-liberation
!pip install -q -r requirements.txt faster-whisper demucs
print('✅ Ambiente GPU pronto!')

In [ ]:
#@title 3. Carregar Chaves Secretas (Colab Secrets ou `.env` privado salvo no seu Drive)
# Dica: salve um arquivo `.env` em `Meu Drive/TwitchAutoClipper/.env` UMA única vez
# para nunca mais precisar digitar chaves ao abrir o Colab!
import os
from pathlib import Path
from dotenv import load_dotenv

drive_env = Path('/content/drive/MyDrive/TwitchAutoClipper/.env')
if drive_env.exists():
    load_dotenv(drive_env, override=True)
    print(f'✅ Credenciais carregadas automaticamente de {drive_env}')
else:
    try:
        from google.colab import userdata
        for k in ['TWITCH_CLIENT_ID', 'TWITCH_CLIENT_SECRET', 'GEMINI_API_KEY', 'GROQ_API_KEY', 'DISCORD_BOT_TOKEN', 'DISCORD_CHANNEL_ID', 'IG_ACCOUNT_ID', 'IG_ACCESS_TOKEN', 'TIKTOK_ACCESS_TOKEN']:
            try:
                val = userdata.get(k)
                if val:
                    os.environ[k] = val
            except Exception:
                pass
        print('✅ Credenciais verificadas via Google Colab Secrets (ícone de chave na barra lateral).')
    except Exception as e:
        print('⚠️ Crie `/content/drive/MyDrive/TwitchAutoClipper/.env` ou adicione as chaves no Colab Secrets:', e)

os.environ['GDRIVE_MOUNT_PATH'] = '/content/drive/MyDrive/TwitchAutoClipper'

In [ ]:
#@title 4. 🚀 Rodada Instantânea em Lote com GPU (Extrair + Legendagem OpusClip + Enviar ao Discord)
PERFIL_OPCIONAL = '' #@param {type:"string"}
MAX_CORTES_AGORA = 10 #@param {type:"slider", min:1, max:15, step:1}

%cd /content/twitch-autocliper
cmd = f'python -m src.main watch --max-cuts {MAX_CORTES_AGORA}'
if PERFIL_OPCIONAL.strip():
    cmd += f' --profile {PERFIL_OPCIONAL.strip()}'
!{cmd}

In [ ]:
#@title 5. 🔇 Limpar Música com Direitos Autorais (Demucs GPU em 5s)
CUT_ID = '' #@param {type:"string"}

%cd /content/twitch-autocliper
if not CUT_ID.strip():
    print('Informe o ID do corte (visível no rodapé do card do Discord).')
else:
    !python -m src.main discord-action --action clean_audio --cut-id {CUT_ID.strip()}

In [ ]:
#@title 6. ♾️ Modo Live Maratona em Segundo Plano (até 24h com aba fechada no Colab Ultra)
INTERVALO_MINUTOS = 8 #@param {type:"slider", min:3, max:30, step:1}
DURACAO_MAXIMA_HORAS = 12 #@param {type:"slider", min:1, max:23, step:1}

%cd /content/twitch-autocliper
!python -m src.main colab-loop --interval-minutes {INTERVALO_MINUTOS} --max-hours {DURACAO_MAXIMA_HORAS}